In [ ]:
!pip install numpy pandas scikit-learn joblib skl2onnx onnx
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import joblib

# 1. Load and clean dataset
try:
    df = pd.read_csv("household_power_consumption.csv")
except Exception as e:
    raise RuntimeError(f"Failed to load dataset: {e}")

cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity',
        'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df.dropna(inplace=True)

# 2. Feature engineering
features = df[cols].copy()
window = 5
for col in cols:
    features[f"{col}_trend"] = features[col].diff(periods=window).fillna(0) / window

# 3. Predictive hazard labeling
labels = np.zeros(len(features))
labels[(features['Global_active_power_trend'] > 0.5) | (features['Global_intensity_trend'] > 0.5)] = 1
labels[(features['Voltage_trend'] < -0.5) | (features['Voltage_trend'] > 0.5)] = 2

# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# 5. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train model
model = RandomForestClassifier(n_estimators=150, random_state=42)
model.fit(X_train_scaled, y_train)

# 7. Evaluate
acc = model.score(X_test_scaled, y_test)
print(f"Test accuracy: {acc:.2f}")
print("Sample predictions:", model.predict(X_test_scaled[:5]))

# 8. Save model and scaler
joblib.dump(scaler, "scaler_pm_mini_real.joblib")
joblib.dump(model, "pm_mini_model_real.pkl")

# 9. Export to ONNX
initial_type = [('float_input', FloatTensorType([None, X_train_scaled.shape[1]]))]
onnx_model = convert_sklearn(model, initial_types=initial_type)
with open("pm_mini_model_real.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Electricity hazard ONNX model saved as pm_mini_model_real.onnx")








Test accuracy: 1.00
Shelly PM Mini ONNX model saved as pm_mini_model_real.onnx


In [ ]:
import pandas as pd

# 1️⃣ Load the text file
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=';',
    low_memory=False,   # avoids dtype warnings
    na_values=['?', ''] # treat missing values as NaN
)

# 2️⃣ Convert numeric columns to floats (skip non-numeric)
numeric_cols = ['Global_active_power', 'Global_reactive_power',
                'Voltage', 'Global_intensity',
                'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3️⃣ Optionally, drop rows with missing numeric values
df.dropna(subset=numeric_cols, inplace=True)

# 4️⃣ Save as CSV
df.to_csv("household_power_consumption.csv", index=False)

print("Conversion complete! Clean CSV saved as 'household_power_consumption.csv'.")


Conversion complete! Clean CSV saved as 'household_power_consumption.csv'.
